In [4]:
#This notebook is for modeling and evaluating the Averitec dataset
from pathlib import Path
import pandas as pd

In [5]:
BASE_DIR = Path("..").resolve()

averitec = pd.read_csv(BASE_DIR / "data" / "processed" / "averitec_cleaned.csv")

averitec.head()

,claim,required_reannotation,label,justification,claim_date,speaker,original_claim_url,fact_checking_article,reporting_source,location_ISO_code,claim_types,fact_checking_strategies,questions,cached_original_claim_url,verdict
0,"In a letter to Steve Jobs, Sean Connery refuse...",False,Refuted,The answer and sources show that the claim was...,31-10-2020,Unknown,NaN,https://web.archive.org/web/20201130144023/htt...,Facebook,NaN,['Event/Property Claim'],['Written Evidence'],[{'question': 'Where was the claim first publi...,NaN,false
1,Trump Administration claimed songwriter Billie...,False,Refuted,Seems that the Wzshington post accused the sin...,31-10-2020,Unknown,NaN,https://web.archive.org/web/20201103001419/htt...,Instagram,US,"['Position Statement', 'Event/Property Claim']",['Written Evidence'],[{'question': 'Has the Trump administration vo...,NaN,false
2,Due to Imran Khan's criticism of Macron's comm...,False,Refuted,The tweet was not the official government page...,31-10-2020,Consulate General Of Pakistan France,https://web.archive.org/web/20201113115127/htt...,https://web.archive.org/web/20210629013122/htt...,Twitter,FR,"['Causal Claim', 'Event/Property Claim']",['Written Evidence'],[{'question': 'How did Macron criticise Islam?...,https://web.archive.org/web/20201113115127/htt...,false
3,UNESCO declared Nadar community as the most an...,False,Refuted,This claim is refuted. According to the QA pai...,31-10-2020,Kumar Shankar,NaN,https://web.archive.org/web/20210225110220/htt...,Facebook,IN,['Event/Property Claim'],['Written Evidence'],"[{'question': 'What is Nadar?', 'answers': [{'...",NaN,false
4,Republican Matt Gaetz was part of a company th...,True,Refuted,The company was sold in 2004 and the law suit ...,31-10-2020,Unknown,NaN,https://web.archive.org/web/20210713185816/htt...,Facebook,US,"['Numerical Claim', 'Event/Property Claim']","['Written Evidence', 'Numerical Comparison']",[{'question': 'Did Matt Gaetz work for Chemed ...,NaN,false


In [7]:
claim_types = averitec["claim_types"].value_counts()
print(claim_types)

claim_types
['Event/Property Claim']                                                245
['Numerical Claim']                                                      80
['Quote Verification']                                                   55
['Numerical Claim', 'Event/Property Claim']                              31
['Causal Claim']                                                         24
['Causal Claim', 'Event/Property Claim']                                 20
['Position Statement']                                                   20
['Quote Verification', 'Event/Property Claim']                            6
['Causal Claim', 'Numerical Claim']                                       6
['Position Statement', 'Event/Property Claim']                            3
['Causal Claim', 'Quote Verification']                                    3
['Position Statement', 'Quote Verification']                              3
['Numerical Claim', 'Quote Verification', 'Event/Property Claim']         1


In [ ]:
def serialize_claim_types(claim_types_str):
    result = (
        claim_types_str.replace("[", "")
            .replace("]", "")
            .replace("'", "")
            .strip()
            .replace("/", '_or_')
            .replace(" ", "_")
            .replace(",_", ",")
            .lower()
    )
    return [claim_type.strip() for claim_type in result.split(",")]

claim_types_set = set()
for idx in claim_types.index:
    result = serialize_claim_types(idx)
    for claim_type in result:
        claim_types_set.add(claim_type)

claim_type_columns = []
for claim_type in claim_types_set:
    column_name = f"is_{claim_type}"
    claim_type_columns.append(column_name)

claim_type_averitec = averitec.copy()

# Create all one-hot columns up front to avoid KeyError
for column_name in claim_type_columns:
    claim_type_averitec[column_name] = 0.0

for idx, row in claim_type_averitec.iterrows():
    claim_types_str = row["claim_types"]
    claim_types_list = set(serialize_claim_types(claim_types_str))
    for claim_type in claim_types_list:
        column_name = f"is_{claim_type}"
        if column_name in claim_type_averitec.columns:
            claim_type_averitec.at[idx, column_name] = 1.0

claim_type_averitec[claim_type_columns] = claim_type_averitec[claim_type_columns].astype(float)
claim_type_averitec.head()



{'quote_verification', 'position_statement', 'event_or_property_claim', 'causal_claim', 'numerical_claim'}


,claim,required_reannotation,label,justification,claim_date,speaker,original_claim_url,fact_checking_article,reporting_source,location_ISO_code,...,claim_type_quote_verification,claim_type_position_statement,claim_type_event_or_property_claim,claim_type_causal_claim,claim_type_numerical_claim,is_quote_verification,is_position_statement,is_event_or_property_claim,is_causal_claim,is_numerical_claim
0,"In a letter to Steve Jobs, Sean Connery refuse...",False,Refuted,The answer and sources show that the claim was...,31-10-2020,Unknown,NaN,https://web.archive.org/web/20201130144023/htt...,Facebook,NaN,...,0,0,0,0,0,0.0,0.0,1.0,0.0,0.0
1,Trump Administration claimed songwriter Billie...,False,Refuted,Seems that the Wzshington post accused the sin...,31-10-2020,Unknown,NaN,https://web.archive.org/web/20201103001419/htt...,Instagram,US,...,0,0,0,0,0,0.0,1.0,1.0,0.0,0.0
2,Due to Imran Khan's criticism of Macron's comm...,False,Refuted,The tweet was not the official government page...,31-10-2020,Consulate General Of Pakistan France,https://web.archive.org/web/20201113115127/htt...,https://web.archive.org/web/20210629013122/htt...,Twitter,FR,...,0,0,0,0,0,0.0,0.0,1.0,1.0,0.0
3,UNESCO declared Nadar community as the most an...,False,Refuted,This claim is refuted. According to the QA pai...,31-10-2020,Kumar Shankar,NaN,https://web.archive.org/web/20210225110220/htt...,Facebook,IN,...,0,0,0,0,0,0.0,0.0,1.0,0.0,0.0
4,Republican Matt Gaetz was part of a company th...,True,Refuted,The company was sold in 2004 and the law suit ...,31-10-2020,Unknown,NaN,https://web.archive.org/web/20210713185816/htt...,Facebook,US,...,0,0,0,0,0,0.0,0.0,1.0,0.0,1.0
